In [ ]:
stat -f "%SB" "032_Preparing to Model.ipynb"

In [ ]:
library(tidyverse)
library(lme4)
library(lmerTest)
library(broom)
library(broom.mixed)
library(ggeffects)
library(emmeans)
library(performance)
library(patchwork)
library(cowplot)
library(modelsummary)
# source("scripts/simulate_lmm_data.R")

In [ ]:
simulate_lmm_data <- function() {
 
set.seed(123)
 
n_subjects <- 30
n_time <- 6
 
subject_df <- tibble(
  subject = factor(1:n_subjects),
  b0 = rnorm(n_subjects, mean = 0, sd = 6),
  b1 = rnorm(n_subjects, mean = 0, sd = 1.2)
)
 
df <- expand_grid(
  subject = factor(1:n_subjects),
  time = 0:(n_time - 1)
) |>
  left_join(subject_df, by = "subject") |>
  mutate(
    y = 20 + 2.5 * time + b0 + b1 * time + rnorm(n(), mean = 0, sd = 2)
  )
  return(df)
  }

In [ ]:
# Organize Function (equivalent of dplyr's arrange function). From https://stackoverflow.com/questions/75667332/base-analogue-of-arrange-in-pipelines
organize <- function (df, ..., na.last = TRUE, decreasing = FALSE,
                      method = c("auto", "shell", "radix"), drop = FALSE) {
  method <- match.arg(method)
  dots <- eval(substitute(alist(...)))
  exprs <- lapply(dots, FUN = \(e) eval(e, df, parent.frame())) 
  arg_ls <- c(exprs,
              na.last = na.last,
              decreasing = decreasing,
              method = method
              )
  df[do.call("order", arg_ls), , drop = drop]
}

In [ ]:
tbl1 <- read.csv("030_ADNI Merge Cohort V5")

# Preparing to Model

In [ ]:
# Taking only one row per participants
tbl2 <- tbl1 |>
    distinct(RID, .keep_all = TRUE)

In [ ]:
# Dividing participants into IR tertiles
tbl3 <- tbl2 |>
    # Dividing TG/HDL Ratio into tertiles
    transform(TG_HDL_Tertiles = cut(TG_HDL_Ratio, breaks = quantile(TG_HDL_Ratio, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) |>
    # Dividing TYG index into tertiles
    transform(TYG_Tertiles = cut(TYG, breaks = quantile(TYG, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) |>
    # Dividing TYG-BMI index into tertiles
    transform(TYG_BMI_Tertiles = cut(TYG_BMI, breaks = quantile(TYG_BMI, probs = (0:3)/3, na.rm = TRUE), 
                                    labels = c("T1", "T2", "T3"),
                                    include.lowest = TRUE)) |>
    # Dividing METS-IR into tertiles
    transform(METS_IR_Tertiles = cut(METS_IR, breaks = quantile(METS_IR, probs = (0:3)/3, na.rm = TRUE), 
                                     labels = c("T1", "T2", "T3"),
                                     include.lowest = TRUE))
tbl3

In [ ]:
# TG/HDL Tertiles: T1 = [0.432, 1.27], T2 = (1.27, 2.16], T3 = (2.16, 7.8]
# TYG Tertiles: T1 = [7.32, 8.18], T2 = (8.18, 8.61], T3 = (8.61, 10]
# TYG-BMI Tertiles: T1 = [126, 204], T2 = (204, 234], T3 = (234, 476]
# METS-IR Tertiles: T1 = [20.3, 34], T2 = (34, 39.6], T3 = (39.6, 79.4]

In [ ]:
# Merging tertile data into cohort dataframe
tbl4 <- tbl3[, colnames(tbl3) %in% c("RID", "TG_HDL_Tertiles", "TYG_Tertiles", "TYG_BMI_Tertiles", "METS_IR_Tertiles")]
tbl5 <- merge(tbl1, tbl4, by = "RID")

In [ ]:
# Calculating Time Difference (in days) between Neurobiomarker exam date & IR_DATE
    # Function to convert VISCODE to numeric
    viscode_to_months <- function(x) {ifelse(x == "sc", -1, ifelse(x == "bl", 0, as.numeric(sub("m", "", x))))}
colnames(tbl5)

In [ ]:
tbl6 <- tbl5 |>
    mutate(EXAMDATE = as.Date(EXAMDATE)) |>
    mutate(EXAMDATE_bl = as.Date(EXAMDATE_bl)) |>
    mutate(IR_DATE = as.Date(IR_DATE)) |>
    mutate(VISCODEnum = viscode_to_months(VISCODE))
tbl7 <- tbl6 |>
    mutate(TIMEDIFF_days = EXAMDATE - IR_DATE)

In [ ]:
sum(is.na(tbl7$VISCODEnum))
# Not sure about missing values

In [ ]:
# Cutting down so only rows with biomarker values are saved
tbl8 <- tbl7 |>
    subset(ABETA > 0) |>
    subset(TAU > 0) |>
    subset(PTAU > 0)

In [ ]:
tbl9 <- tbl8[, colnames(tbl8) %in% c("RID", "COLPROT", "ORIGPROT", "PTID", "SITE", 
                                     "VISCODE", "EXAMDATE", "DX_bl", "AGE", "PTGENDER",
                                     "PTEDUCAT", "PTETHCAT", "PTRACCAT", "PTMARRY", "APOE4",
                                     "ABETA", "TAU", "PTAU",
                                     "CDR", "ADAS11", "ADAS13", "MMSE", "MOCA", "EcogPtTotal", "EcogSPTotal",
                                     "DX", 
                                     "EXAMDATE_bl", "CDR_bl", "ADAS11_bl", "ADAS13_bl", "MMSE_bl", "MOCA_bl", "EcogPtTotal_bl", "EcogSPTotal_bl",
                                     "ABETA_bl", "TAU_bl", "PTAU_bl",
                                     "Years_bl", "Month_bl", "Month", 
                                     "IR_DATE", "TG_HDL_Ratio", "TYG", "TYG_BMI", "METS_IR", 
                                     "TG_HDL_Tertiles", "TYG_Tertiles", "TYG_BMI_Tertiles", "METS_IR_Tertiles",
                                     "MEDHIST_DATE", "CVD", "ALCOHOL", "DRUG_AB", "SMOKING", "SUBSTANCE_AB",
                                     "BMIHT_DATE", "BMI", "HYPERTENSION",
                                     "metabolic_DATE", "HDL_C", "TOTAL_TG", "GLUCOSE", "DIABETES",
                                     "VISCODEnum", "TIMEDIFF_days")]

In [ ]:
# Counting Participants and datapoints
aggregate1 <- aggregate(ABETA ~ RID, tbl9, length)
nrow(aggregate1) # 907 participants
nrow(tbl9) # 1836 datapoints

In [ ]:
# Finding participants with a bl visit
tbl10 <- tbl9 |>
    subset(VISCODE == "bl")
tbl10 <- tbl10[, colnames(tbl10) %in% c("RID", "VISCODE")]

In [ ]:
# Keeping only participants with a bl visit
tbl11 <- merge(tbl9, tbl10, by = "RID")
tbl11 <- tbl11 |>
    rename(VISCODE = VISCODE.x)

In [ ]:
# Removing VISCODE.y column
tbl12 <- tbl11[, colnames(tbl11) %in% c("RID", "COLPROT", "ORIGPROT", "PTID", "SITE", 
                                     "VISCODE", "EXAMDATE", "DX_bl", "AGE", "PTGENDER",
                                     "PTEDUCAT", "PTETHCAT", "PTRACCAT", "PTMARRY", "APOE4",
                                     "ABETA", "TAU", "PTAU",
                                     "CDR", "ADAS11", "ADAS13", "MMSE", "MOCA", "EcogPtTotal", "EcogSPTotal",
                                     "DX", 
                                     "EXAMDATE_bl", "CDR_bl", "ADAS11_bl", "ADAS13_bl", "MMSE_bl", "MOCA_bl", "EcogPtTotal_bl", "EcogSPTotal_bl",
                                     "ABETA_bl", "TAU_bl", "PTAU_bl",
                                     "Years_bl", "Month_bl", "Month", 
                                     "IR_DATE", "TG_HDL_Ratio", "TYG", "TYG_BMI", "METS_IR", 
                                     "TG_HDL_Tertiles", "TYG_Tertiles", "TYG_BMI_Tertiles", "METS_IR_Tertiles",
                                     "MEDHIST_DATE", "CVD", "ALCOHOL", "DRUG_AB", "SMOKING", "SUBSTANCE_AB",
                                     "BMIHT_DATE", "BMI", "HYPERTENSION",
                                     "metabolic_DATE", "HDL_C", "TOTAL_TG", "GLUCOSE", "DIABETES",
                                     "VISCODEnum", "TIMEDIFF_days")]

In [ ]:
# Counting Participants and datapoints
aggregate2 <- aggregate(ABETA ~ RID, tbl12, length)
nrow(aggregate2) # 877 participants
nrow(tbl12) # 1779 datapoints

In [ ]:
# Changing ABETA & PTAU to numeric (TAU is already numeric)

#ABETA
table(tbl12$ABETA)
    # <200 -> 200
    # >1700 -> 1700
    # Compresses extremes a bit

In [ ]:
# Cleaning ABETA column
f_clean_numeric <- function(x) {as.numeric(gsub("[^0-9.]", "", x))}
tbl12$ABETA_clean <- f_clean_numeric(tbl12$ABETA)

In [ ]:
#PTAU
table(tbl12$PTAU)
    # <8 -> 8
f_clean_numeric <- function(x) {as.numeric(gsub("[^0-9.]", "", x))}
tbl12$PTAU_clean <- f_clean_numeric(tbl12$PTAU)

In [ ]:
# Saving Cohort
write.csv(tbl12, "033_ADNI Merge Cohort Final")